<a href="https://colab.research.google.com/github/rymarinelli/Advanced-Programming-/blob/main/PINN_December.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! gdown 1VFOQkLJcorQ0VDWV7urD8ytSIrumAcBp -O clearsky.iq
! gdown 1Nz4mVVp61Kbq2nwsWije1SyH-7h3qnsC -O jammed.iq

Downloading...
From (original): https://drive.google.com/uc?id=1VFOQkLJcorQ0VDWV7urD8ytSIrumAcBp
From (redirected): https://drive.google.com/uc?id=1VFOQkLJcorQ0VDWV7urD8ytSIrumAcBp&confirm=t&uuid=cbbb7181-295e-4320-ac26-f70ad2dceb97
To: /content/clearsky.iq
100% 2.32G/2.32G [00:32<00:00, 71.3MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Nz4mVVp61Kbq2nwsWije1SyH-7h3qnsC
From (redirected): https://drive.google.com/uc?id=1Nz4mVVp61Kbq2nwsWije1SyH-7h3qnsC&confirm=t&uuid=6e766244-5a94-4abc-bfc9-4be479be8b5e
To: /content/jammed.iq
100% 9.26G/9.26G [02:25<00:00, 63.6MB/s]


In [1]:
# =========================================
# PirateNet Residual Fourier PINN
# Real IQ file loading + extraction %
# =========================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# -------------------------------
# IQ FILE LOADING
# -------------------------------

def load_iq(path, dtype=np.float32):
    raw = np.fromfile(path, dtype=dtype)
    iq = raw[0::2] + 1j * raw[1::2]
    return torch.tensor(iq, dtype=torch.complex64)

# -------------------------------
# METRICS
# -------------------------------

def normalize_power(x):
    return x / torch.sqrt(torch.mean(torch.abs(x)**2) + 1e-8)

def extraction_percentage(y, x_hat):
    residual = y - x_hat
    return (1 - torch.sum(torch.abs(residual)**2) /
                torch.sum(torch.abs(y)**2)).item() * 100

def spectral_extraction_percentage(y, x_hat, fs):
    Y = torch.fft.fft(y)
    R = torch.fft.fft(y - x_hat)
    freqs = torch.fft.fftfreq(len(y), d=1/fs)
    jam_mask = torch.abs(freqs) > 0.2 * fs
    return (1 - torch.sum(torch.abs(R[jam_mask])**2) /
                torch.sum(torch.abs(Y[jam_mask])**2)).item() * 100

def maxwell_violation(E_hat, B_hat, k, omega):
    return torch.mean(torch.abs(omega * B_hat - k * E_hat)**2 +
                      torch.abs(k * E_hat)**2)

# -------------------------------
# MODEL
# -------------------------------

class FourierResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mag = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Linear(dim, dim)
        )
        self.phase = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Linear(dim, dim)
        )

    def forward(self, mag, phase):
        return mag + self.mag(mag), phase + self.phase(phase)

class PirateNet(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.frb1 = FourierResidualBlock(dim)
        self.frb2 = FourierResidualBlock(dim)
        self.E = nn.Linear(dim, dim)
        self.B = nn.Linear(dim, dim)

    def forward(self, X):
        mag, phase = torch.abs(X), torch.angle(X)
        mag, phase = self.frb1(mag, phase)
        mag, phase = self.frb2(mag, phase)
        Xc = mag * torch.exp(1j * phase)
        E = self.E(Xc.real) + 1j * self.E(Xc.imag)
        B = self.B(Xc.real) + 1j * self.B(Xc.imag)
        return Xc, E, B

# -------------------------------
# TRAINING
# -------------------------------

def train_step(model, y, fs, omega, lambda_phys, opt):
    y = normalize_power(y)
    X = torch.fft.fft(y)

    freqs = torch.fft.fftfreq(len(y), d=1/fs)
    k = 2 * torch.pi * freqs / 3e8

    Xc, E, B = model(X)
    y_hat = torch.fft.ifft(Xc)

    L_data = torch.mean(torch.abs(y - y_hat)**2)
    L_phys = maxwell_violation(E, B, k, omega)

    loss = L_data + lambda_phys * L_phys
    opt.zero_grad()
    loss.backward()
    opt.step()

    return (
        loss.item(),
        extraction_percentage(y, y_hat),
        spectral_extraction_percentage(y, y_hat, fs),
        L_phys.item()
    )

# -------------------------------
# MAIN
# -------------------------------

if __name__ == "__main__":

    fs = 1_000_000          # ⚠️ set correct sample rate
    fc = 100_000            # ⚠️ approximate carrier
    omega = 2 * torch.pi * fc

    clean = load_iq("/content/clearsky.iq")
    jammed = load_iq("/content/jammed.iq")

    N = min(len(clean), len(jammed))
    clean, jammed = clean[:N], jammed[:N]

    model = PirateNet(N)
    opt = optim.Adam(model.parameters(), lr=1e-3)

    print("Training PirateNet...\n")

    for epoch in range(1, 201):

        if epoch < 50:
            λp = 0.0
        elif epoch < 120:
            λp = 0.01
        else:
            λp = 0.1

        loss, et, ef, lp = train_step(
            model, jammed, fs, omega, λp, opt
        )

        if epoch % 10 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Loss {loss:.3e} | "
                f"Extract {et:.2f}% | "
                f"Spectral {ef:.2f}% | "
                f"Maxwell {lp:.2e}"
            )

    # Final evaluation vs clean reference
    with torch.no_grad():
        Xc, _, _ = model(torch.fft.fft(jammed))
        extracted = torch.fft.ifft(Xc)

        ref_gain = 1 - torch.sum(torch.abs(clean - extracted)**2) / \
                       torch.sum(torch.abs(jammed - clean)**2)

        print(f"\n✅ True jammer removal vs clean: {ref_gain.item()*100:.2f}%")


In [ ]:
def window_iq(x, window_size, hop_size):
    """
    Splits IQ signal into overlapping windows.
    """
    windows = []
    for start in range(0, len(x) - window_size + 1, hop_size):
        windows.append(x[start:start + window_size])
    return torch.stack(windows)


In [ ]:
def normalize_window(x):
    return x / torch.sqrt(torch.mean(torch.abs(x)**2) + 1e-8)


In [ ]:
WINDOW = 2048
HOP = 1024

clean_w = window_iq(clean, WINDOW, HOP)
jammed_w = window_iq(jammed, WINDOW, HOP)

num_windows = min(len(clean_w), len(jammed_w))
clean_w = clean_w[:num_windows]
jammed_w = jammed_w[:num_windows]


In [ ]:
# ============================================================
# PirateNet: Windowed Residual Fourier PINN (Corrected)
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# -------------------------------
# CONFIG
# -------------------------------
FS = 1_000_000          # Sample rate (Hz)
FC = 100_000            # Carrier estimate (Hz)
SECONDS = 5
WINDOW = 2048
HOP = 1024
EPOCHS = 200
LR = 1e-3

omega = 2 * torch.pi * FC

# -------------------------------
# IQ LOADING
# -------------------------------
def load_iq(path):
    raw = np.fromfile(path, dtype=np.float32)
    iq = raw[0::2] + 1j * raw[1::2]
    return torch.tensor(iq, dtype=torch.complex64)

# -------------------------------
# WINDOWING + NORMALIZATION
# -------------------------------
def window_iq(x, win, hop):
    return torch.stack([
        x[i:i+win] for i in range(0, len(x) - win + 1, hop)
    ])

def normalize_window(x):
    return x / torch.sqrt(torch.mean(torch.abs(x)**2) + 1e-8)

# -------------------------------
# METRICS
# -------------------------------
def extraction_percentage(y, x_hat):
    return (1 - torch.sum(torch.abs(y - x_hat)**2) /
                torch.sum(torch.abs(y)**2)).item() * 100

def spectral_extraction_percentage(y, x_hat):
    Y = torch.fft.fft(y)
    R = torch.fft.fft(y - x_hat)
    freqs = torch.fft.fftfreq(len(y), d=1/FS)
    jam_mask = torch.abs(freqs) > 0.2 * FS
    return (1 - torch.sum(torch.abs(R[jam_mask])**2) /
                torch.sum(torch.abs(Y[jam_mask])**2)).item() * 100

def maxwell_violation(E, B, k, eps=1e-8):
    t1 = omega * B
    t2 = k * E
    t1 = t1 / (torch.mean(torch.abs(t1)) + eps)
    t2 = t2 / (torch.mean(torch.abs(t2)) + eps)
    return torch.mean(torch.abs(t1 - t2)**2)

# -------------------------------
# MODEL
# -------------------------------
class FourierResidualBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.mag = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Linear(d, d)
        )
        self.phase = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Linear(d, d)
        )

    def forward(self, mag, phase):
        dmag = self.mag(mag)
        dphi = self.phase(phase)

        # gated, relative magnitude update
        mag = mag * (1.0 + 0.05 * torch.tanh(dmag))
        phase = phase + dphi
        return mag, phase

class PirateNet(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.frb1 = FourierResidualBlock(d)
        self.frb2 = FourierResidualBlock(d)
        self.E = nn.Linear(d, d)
        self.B = nn.Linear(d, d)

    def forward(self, X):
        mag, phase = torch.abs(X), torch.angle(X)
        mag, phase = self.frb1(mag, phase)
        mag, phase = self.frb2(mag, phase)
        Xc = mag * torch.exp(1j * phase)
        E = self.E(Xc.real) + 1j * self.E(Xc.imag)
        B = self.B(Xc.real) + 1j * self.B(Xc.imag)
        return Xc, E, B

# -------------------------------
# LOAD DATA (5 SECONDS)
# -------------------------------
clean = load_iq("/content/clearsky.iq")
jammed = load_iq("/content/jammed.iq")

N = int(FS * SECONDS)
clean = clean[:N]
jammed = jammed[:N]

clean_w = window_iq(clean, WINDOW, HOP)
jammed_w = window_iq(jammed, WINDOW, HOP)

num_windows = min(len(clean_w), len(jammed_w))
clean_w = clean_w[:num_windows]
jammed_w = jammed_w[:num_windows]

print(f"Training on {num_windows} windows (~{SECONDS}s)")

# -------------------------------
# TRAINING
# -------------------------------
model = PirateNet(WINDOW)
opt = optim.Adam(model.parameters(), lr=LR)

freqs = torch.fft.fftfreq(WINDOW, d=1/FS)
k = 2 * torch.pi * freqs / 3e8

for epoch in range(1, EPOCHS + 1):

    # DELAYED PHYSICS CURRICULUM
    if epoch < 120:
        lambda_phys = 0.0
    elif epoch < 180:
        lambda_phys = 0.005
    else:
        lambda_phys = 0.02

    et_list, ef_list = [], []

    pbar = tqdm(jammed_w, desc=f"Epoch {epoch:03d}", leave=False)

    for y_win in pbar:
        y_win = normalize_window(y_win)
        X = torch.fft.fft(y_win)

        Xc, E, B = model(X)
        y_hat = torch.fft.ifft(Xc)

        L_data = torch.mean(torch.abs(y_win - y_hat)**2)
        L_phys = maxwell_violation(E, B, k)

        # ENERGY PRESERVATION (critical)
        L_energy = torch.abs(
            torch.mean(torch.abs(y_hat)) -
            torch.mean(torch.abs(y_win))
        )

        loss = L_data + lambda_phys * L_phys + 0.05 * L_energy

        opt.zero_grad()
        loss.backward()
        opt.step()

        et = extraction_percentage(y_win, y_hat)
        ef = spectral_extraction_percentage(y_win, y_hat)

        et_list.append(et)
        ef_list.append(ef)

        pbar.set_postfix(
            extract=f"{et:.1f}%",
            spec=f"{ef:.1f}%",
            phys=f"{L_phys.item():.2e}"
        )

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Extract {np.mean(et_list):.2f}% | "
            f"Spectral {np.mean(ef_list):.2f}% | "
            f"λ_phys {lambda_phys}"
        )

# -------------------------------
# TRUE EXTRACTION VS CLEAR SKY
# -------------------------------
with torch.no_grad():
    gains = []
    for c, j in zip(clean_w, jammed_w):
        c = normalize_window(c)
        j = normalize_window(j)
        Xc, _, _ = model(torch.fft.fft(j))
        extracted = torch.fft.ifft(Xc)
        gain = 1 - torch.sum(torch.abs(c - extracted)**2) / \
                   torch.sum(torch.abs(j - c)**2)
        gains.append(gain.item() * 100)

print(
    f"\n✅ FINAL 5s TRUE EXTRACTION: "
    f"{np.mean(gains):.2f}% ± {np.std(gains):.2f}%"
)
